In [ ]:
# image_clustering_pipeline.py

import torch
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
import umap
import hdbscan
from sklearn.metrics import silhouette_score
from typing import Tuple, Dict, Any
import matplotlib.pyplot as plt
import pandas as pd

def torch_to_numpy(embeddings: torch.Tensor) -> np.ndarray:
    """Convert torch tensor to NumPy array (float32)."""
    if embeddings.is_cuda:
        embeddings = embeddings.cpu()
    return embeddings.detach().numpy().astype(np.float32)

def normalize_embeddings(embeddings: np.ndarray) -> np.ndarray:
    """L2 normalize embeddings."""
    return normalize(embeddings, norm='l2')

def reduce_dimensions(
    embeddings: np.ndarray,
    pca_dim: int = 50,
    umap_dim: int = 15,
    umap_neighbors: int = 30,
    umap_min_dist: float = 0.0
) -> np.ndarray:
    """Apply PCA + UMAP reduction."""
    pca = PCA(n_components=pca_dim, random_state=42)
    reduced = pca.fit_transform(embeddings)

    umap_model = umap.UMAP(
        n_neighbors=umap_neighbors,
        n_components=umap_dim,
        min_dist=umap_min_dist,
        metric='cosine',
        random_state=42
    )
    umap_embeddings = umap_model.fit_transform(reduced)
    return umap_embeddings

def cluster_embeddings(
    reduced_embeddings: np.ndarray,
    min_cluster_size: int = 30
) -> Tuple[np.ndarray, hdbscan.HDBSCAN]:
    """Cluster embeddings using HDBSCAN."""
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    cluster_labels = clusterer.fit_predict(reduced_embeddings)
    return cluster_labels, clusterer

def evaluate_clusters(embeddings: np.ndarray, labels: np.ndarray) -> float:
    """Compute silhouette score (ignoring noise)."""
    mask = labels != -1
    if np.sum(mask) > 1:
        score = silhouette_score(embeddings[mask], labels[mask])
        return score
    return -1.0

def run_image_clustering_pipeline(
    torch_embeddings: torch.Tensor,
    min_cluster_size: int = 30
) -> Dict[str, Any]:
    """Full image clustering pipeline."""
    print("✅ Converting embeddings...")
    np_embeddings = torch_to_numpy(torch_embeddings)
    np_embeddings = normalize_embeddings(np_embeddings)

    print("✅ Reducing dimensions...")
    reduced = reduce_dimensions(np_embeddings)

    print("✅ Clustering embeddings...")
    cluster_labels, clusterer = cluster_embeddings(reduced, min_cluster_size)

    print("✅ Evaluating clusters...")
    sil_score = evaluate_clusters(reduced, cluster_labels)
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)

    print(f"✅ Found {n_clusters} clusters (Silhouette: {sil_score:.3f})")

    results = {
        "labels": cluster_labels,
        "clusterer": clusterer,
        "silhouette": sil_score,
        "n_clusters": n_clusters
    }

    return results

def auto_tune_hdbscan(embeddings, cluster_size_range=(10, 200), step=10):
    """
    Automatically find best HDBSCAN min_cluster_size based on silhouette score.
    """
    results = []

    for min_size in range(cluster_size_range[0], cluster_size_range[1] + 1, step):
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_size,
            metric='euclidean',
            cluster_selection_method='eom'
        ).fit(embeddings)

        labels = clusterer.labels_
        mask = labels != -1
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_ratio = np.mean(labels == -1)

        if n_clusters > 1 and np.sum(mask) > 1:
            sil = silhouette_score(embeddings[mask], labels[mask])
        else:
            sil = -1

        results.append({
            "min_cluster_size": min_size,
            "n_clusters": n_clusters,
            "silhouette": sil,
            "noise_ratio": noise_ratio
        })

        print(f"[min_size={min_size}] clusters={n_clusters}, sil={sil:.3f}, noise={noise_ratio:.2f}")

    # Find best config by silhouette + cluster balance
    df = sorted(results, key=lambda x: (x["silhouette"], -x["noise_ratio"]), reverse=True)
    best = df[0]
    print(f"\n✅ Best config: min_cluster_size={best['min_cluster_size']} → "
          f"{best['n_clusters']} clusters (sil={best['silhouette']:.3f})")

    return best, results


In [ ]:
# Custer trip by Time
import pandas as pd
import numpy as np
from hdbscan import HDBSCAN
from sklearn.metrics.pairwise import cosine_similarity


def segment_by_time(df, time_gap_hours=12):
    """
    Split photos into segments based on time gaps.
    """
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["time_diff_hours"] = df["timestamp"].diff().dt.total_seconds() / 3600
    df["trip_id_by_time"] = (df["time_diff_hours"] > time_gap_hours).cumsum()
    return df
## Cluster trip by time and context
def fuse_time_and_embedding(df, time_weight=0.05):
    # Convert timestamp to numeric (days since first photo)
    df["time_numeric"] = (df["timestamp"] - df["timestamp"].min()).dt.total_seconds() / (3600*24)
    time_feature = df["time_numeric"].to_numpy().reshape(-1, 1) * time_weight
    fused = np.hstack([df["vectors"].to_numpy().tolist(), time_feature])
    return fused

def detect_trips(df, time_weight=0.05, min_cluster_size=10):
    fused_embeddings = fuse_time_and_embedding(df, time_weight)
    
    clusterer = HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric='euclidean',
        cluster_selection_method='eom'
    ).fit(fused_embeddings)
    
    df["trip_id_by_time_and_context"] = clusterer.labels_
    return df, clusterer

def merge_small_clusters(df, cluster_columns_name, min_trip_size=15, sim_threshold=0.85):
    trips = []
    for tid, group in df.groupby("{cluster_columns_name}"):
        if len(group) < min_trip_size:
            trips.append((tid, group))
    # compute centroids
    centroids = {
        tid: np.mean(np.vstack(g["vectors"]), axis=0)
        for tid, g in df.groupby("{cluster_columns_name}")
    }
    # merge logic
    merged_map = {}
    for tid, group in trips:
        best_match = None
        best_sim = -1
        for tid2, cent in centroids.items():
            if tid == tid2: continue
            sim = cosine_similarity(centroids[tid].reshape(1,-1), cent.reshape(1,-1))[0][0]
            if sim > sim_threshold: best_match = tid2; best_sim = sim
        if best_match:
            merged_map[tid] = best_match

    df["{cluster_columns_name}_merged"] = df["{cluster_columns_name}"].replace(merged_map)
    return df
